# **Content-based Course Recommender System: User Profile and Course Genre Analysis**


## Introduction

In this project, I explore content-based recommendation systems for online courses. Content-based recommendation systems suggest items to users based on their established preferences and tastes. These user profiles are developed through various user interactions such as ratings, clicks, and engagement patterns.

The recommendation process leverages similarity between items, measured by analyzing content attributes like category, tags, and genres - essentially the defining features of each item. For online courses, this means extracting meaningful features from courses and building accurate user profiles to match users with relevant educational content.


## The Approach

My implementation focuses on two key components:

1. **User Profile Generation**: Using course genres and user ratings to mathematically construct feature vectors representing a user's learning interests
2. **Course Recommendation**: Employing computational methods like dot products between user profiles and course features to predict interest scores

This approach allows for personalized course recommendations based on demonstrated user preferences.

## Implementation


In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing

# Setting random state for reproducibility
rs = 123

### Generating User Profiles from Course Genres and Ratings


I'll start with a simplified dataset containing three primary genres: `Python`, `Database`, and `MachineLearning`. The dataset includes two courses with their respective genre classifications:

In [2]:
course_genres = ['Python', 'Database', 'MachineLearning']
courses = [['Machine Learning with Python', 1, 0, 1], ["SQL with Python", 1, 1, 0]]
courses_df = pd.DataFrame(courses, columns = ['Title'] + course_genres)
courses_df

,Title,Python,Database,MachineLearning
0,Machine Learning with Python,1,0,1
1,SQL with Python,1,1,0


From this dataset:
- `Machine Learning with Python` is tagged with `Python` and `MachineLearning` genres
- `SQL with Python` is tagged with `Python` and `Database` genres


Next, I'll create a user rating dataset with two users and their course interactions:

In [3]:
users = [['user0', 'Machine Learning with Python', 3], ['user1', 'SQL with Python', 2]]
users_df = pd.DataFrame(users, columns = ['User', 'Title', 'Rating'])
users_df

,User,Title,Rating
0,user0,Machine Learning with Python,3
1,user1,SQL with Python,2


In this dataset:
- `user0` rated `Machine Learning with Python` as 3 (completed with a certificate)
- `user1` rated `SQL with Python` as 2 (audited or not completed)

The question now becomes: Can I generate accurate profile vectors for each user based on these course ratings and genre classifications?

### Mathematical Approach to User Profiling

Intuitively, since `user0` completed the `Machine Learning with Python` course, they likely have interest in Python and Machine Learning topics. Conversely, the absence of engagement with `SQL with Python` suggests potentially lower interest in database topics.

To quantify these interests mathematically, I can multiply the user's rating vector with the course genre matrix to derive a weighted genre vector:

In [4]:
# User 0 rated course 0 as 3 and course 1 as 0/NA (unknown or not interested)
u0 = np.array([[3, 0]])

In [5]:
# The course genre's matrix
C = courses_df[['Python', 'Database', 'MachineLearning']].to_numpy()
C

array([[1, 0, 1],
       [1, 1, 0]])

Examining the dimensions of these matrices:

In [6]:
print(f"User profile vector shape {u0.shape} and course genre matrix shape {C.shape}")

User profile vector shape (1, 2) and course genre matrix shape (2, 3)


When multiplying a $1 \times 2$ vector with a $2 \times 3$ matrix, the result will be a $1 \times 3$ vector representing the user's profile vector:

$$u_0C = \begin{bmatrix} 3 & 0 \end{bmatrix} \begin{bmatrix} 1 & 0 & 1 \\\\\\\\ 1 & 1 & 0 \end{bmatrix}$$

In [7]:
u0_weights = np.matmul(u0, C)
u0_weights

array([[3, 0, 3]])

In [8]:
course_genres

['Python', 'Database', 'MachineLearning']

The resulting `u0_weights` is a weighted genre vector representing the user's interest in each genre. As expected, `user0` shows strong interest in `Python` and `MachineLearning` with a rating of 3 for both.

Similarly, I can calculate the weighted genre vector for `user1`:

$$u_1C = \begin{bmatrix} 0 & 2 \end{bmatrix} \begin{bmatrix} 1 & 0 & 1 \\\\\\\\ 1 & 1 & 0 \end{bmatrix}$$

In [9]:
# User 1 rated course 0 as 0 (unknown or not interested) and course 1 as 2
u1 = np.array([[0, 2]])

In [10]:
u1_weights = np.matmul(u1, C)
u1_weights

array([[2, 2, 0]])

The `u1_weights` vector confirms that `user1` has significant interest in `Python` and `Database` with a value of 2 for each.

Combining these weighted genre vectors into a comprehensive user profile dataframe:

In [11]:
weights = np.concatenate((u0_weights.reshape(1, 3), u1_weights.reshape(1, 3)), axis=0)
profiles_df = pd.DataFrame(weights, columns=['Python', 'Database', 'MachineLearning'])
profiles_df.insert(0, 'user', ['user0', 'user1'])

In [12]:
profiles_df

,user,Python,Database,MachineLearning
0,user0,3,0,3
1,user1,2,2,0


The `profiles_df` clearly illustrates each user's course interests across all genres in our system.

### Generating Personalized Course Recommendations


With user profiles established, I can now see that:
- `user0` has strong interests in Python and machine learning
- `user1` has strong interests in Python and databases

Let's consider three new courses that have been added to the platform:

In [13]:
new_courses = [['Python 101', 1, 0, 0], ["Database 101", 0, 1, 0], ["Machine Learning with R", 0, 0, 1]]
new_courses_df = pd.DataFrame(new_courses, columns = ['Title', 'Python', 'Database', 'MachineLearning'])
new_courses_df

,Title,Python,Database,MachineLearning
0,Python 101,1,0,0
1,Database 101,0,1,0
2,Machine Learning with R,0,0,1


### Calculating Recommendation Scores

To calculate personalized recommendation scores for each new course, I'll apply the dot product between user profile vectors and course genre vectors. Since we have two users and three courses, this requires a matrix multiplication operation:

In [14]:
profiles_df

,user,Python,Database,MachineLearning
0,user0,3,0,3
1,user1,2,2,0


First, converting the course genre dataframe to a 2D numpy array:

In [15]:
# Drop the title column
new_courses_df = new_courses_df.loc[:, new_courses_df.columns != 'Title']
course_matrix = new_courses_df.values
course_matrix

array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]])

In [16]:
# course matrix shape
course_matrix.shape

(3, 3)

The course matrix is a `3 × 3` matrix where each row vector represents a course's genre attributes.

Next, converting the user profile dataframe to another 2D numpy array:

In [17]:
# Drop the user column
profiles_df = profiles_df.loc[:, profiles_df.columns != 'user']
profile_matrix = profiles_df.values
profile_matrix

array([[3, 0, 3],
       [2, 2, 0]])

In [18]:
profile_matrix.shape

(2, 3)

The profile matrix is a `2 × 3` matrix where each row represents a user's profile vector across all genres.

When multiplying the course matrix with the transpose of the user profile matrix, I can generate a `3 × 2` recommendation matrix. Each element `(i, j)` represents the recommendation score of course `i` for user `j`. 

The intuition is straightforward: if a user `j` has demonstrated interest in certain topics (genres) and a course `i` covers those same topics, their respective vectors will have high similarity in those dimensions, resulting in a larger dot product value.

In [19]:
scores = np.matmul(course_matrix, profile_matrix.T)
scores

array([[3, 2],
       [0, 2],
       [3, 0]])

Adding back course titles and user IDs for clarity:

In [20]:
scores_df = pd.DataFrame(scores, columns=['User0', 'User1'])
scores_df.index = ['Python 101', 'Database 101', 'Machine Learning with R']

In [21]:
# recommendation score dataframe
scores_df

,User0,User1
Python 101,3,2
Database 101,0,2
Machine Learning with R,3,0


## Results Analysis

The recommendation scores reveal predictable but valuable patterns:

- For `user0`, the highest scoring courses are `Python 101` (3.0) and `Machine Learning with R` (3.0), aligning perfectly with their demonstrated interest in Python and machine learning topics

- For `user1`, the highest scoring courses are `Python 101` (2.0) and `Database 101` (2.0), matching their established preferences for Python and database subjects

This simple implementation demonstrates how content-based recommender systems can generate personalized course recommendations using linear algebra operations on user profiles and content features.

## Applying the Model to a Real-World Dataset

After developing the theoretical framework for our content-based recommendation system, I'll now apply these techniques to a real-world dataset. This practical implementation will demonstrate how to generate personalized course recommendations for a larger user base using the user profile and course genre vectors approach.


In [22]:
course_genres_df = pd.read_csv("data/course_genre.csv")

In [23]:
course_genres_df.head()

,COURSE_ID,TITLE,Database,Python,CloudComputing,DataAnalysis,Containers,MachineLearning,ComputerVision,DataScience,BigData,Chatbot,R,BackendDev,FrontendDev,Blockchain
0,ML0201EN,robots are coming build iot apps with watson ...,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1,ML0122EN,accelerating deep learning with gpu,0,1,0,0,0,1,0,1,0,0,0,0,0,0
2,GPXX0ZG0EN,consuming restful services using the reactive ...,0,0,0,0,0,0,0,0,0,0,0,1,1,0
3,RP0105EN,analyzing big data in r using apache spark,1,0,0,1,0,0,0,0,1,0,1,0,0,0
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,0,0,0,0,1,0,0,0,0,0,0,1,0,0


In [24]:
profile_df = pd.read_csv("data/user_profile.csv")

In [25]:
profile_df.head()

,user,Database,Python,CloudComputing,DataAnalysis,Containers,MachineLearning,ComputerVision,DataScience,BigData,Chatbot,R,BackendDev,FrontendDev,Blockchain
0,2,52.0,14.0,6.0,43.0,3.0,33.0,0.0,29.0,41.0,2.0,18.0,34.0,9.0,6.0
1,4,40.0,2.0,4.0,28.0,0.0,14.0,0.0,20.0,24.0,0.0,6.0,6.0,0.0,2.0
2,5,24.0,8.0,18.0,24.0,0.0,30.0,0.0,22.0,14.0,2.0,14.0,26.0,4.0,6.0
3,7,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
4,8,6.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,0.0


The profile dataframe contains the course interests for each user. For example, examining user 8's profile reveals strong interests in R, data analysis, database, and big data:

In [26]:
profile_df[profile_df['user'] == 8]

,user,Database,Python,CloudComputing,DataAnalysis,Containers,MachineLearning,ComputerVision,DataScience,BigData,Chatbot,R,BackendDev,FrontendDev,Blockchain
4,8,6.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,6.0,0.0,2.0,0.0,0.0,0.0


Next, I'll load the test dataset containing users for whom I want to generate course recommendations:

In [27]:
test_users_df = pd.read_csv("data/ratings.csv")

In [28]:
test_users_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


Let's identify how many unique test users are in the dataset:

In [29]:
# Extract unique user IDs from the test dataset
test_users = test_users_df.groupby(['user']).max().reset_index(drop=False)
test_user_ids = test_users['user'].to_list()
print(f"Total numbers of test users: {len(test_user_ids)}")

Total numbers of test users: 33901


## Recommendation Process

For each test user, my process involves identifying courses they haven't encountered yet (unknown courses) and calculating recommendation scores for these courses based on their profile. Let me demonstrate this process with a specific user, ID `1078030`:

In [30]:
# Retrieve the profile for user 1078030
test_user_profile = profile_df[profile_df['user'] == 1078030]
test_user_profile

,user,Database,Python,CloudComputing,DataAnalysis,Containers,MachineLearning,ComputerVision,DataScience,BigData,Chatbot,R,BackendDev,FrontendDev,Blockchain
18204,1078030,0.0,12.0,0.0,9.0,0.0,12.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
# Extract the user's interest vector (excluding the 'user' column)
test_user_vector = test_user_profile.iloc[0, 1:].values
test_user_vector

array([ 0., 12.,  0.,  9.,  0., 12.,  0.,  6.,  0.,  0.,  0.,  0.,  0.,
        0.])

First, I need to identify which courses this user has already enrolled in:

In [32]:
enrolled_courses = test_users_df[test_users_df['user'] == 1078030]['item'].to_list()
enrolled_courses = set(enrolled_courses)
enrolled_courses

{'DA0101EN',
 'DV0101EN',
 'ML0101ENv3',
 'ML0115EN',
 'ML0120ENv2',
 'ML0122ENv1',
 'PY0101EN',
 'ST0101EN'}

Next, I'll compile a list of all courses in the system:

In [33]:
all_courses = set(course_genres_df['COURSE_ID'].values)

By finding the difference between all courses and enrolled courses, I can identify the unknown courses for this user - these are potential candidates for recommendations:

In [34]:
unknown_courses = all_courses.difference(enrolled_courses)

For each unknown course, I'll retrieve its genre vector:

In [35]:
unknown_course_genres = course_genres_df[course_genres_df['COURSE_ID'].isin(unknown_courses)]
# Extract the genre features (excluding 'COURSE_ID' and 'TITLE' columns)
course_matrix = unknown_course_genres.iloc[:, 2:].values
course_matrix

array([[0, 0, 0, ..., 1, 1, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 1, 0],
       ...,
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 1, 1, 0],
       [0, 0, 0, ..., 1, 1, 0]])

Using the dot product between the user's profile vector and each course's genre vector, I can calculate personalized recommendation scores. For example, the score for the "accelerating deep learning with gpu" course would be:

In [36]:
score = np.dot(course_matrix[1], test_user_vector)
score

30.0

To determine which courses to actually recommend to users, I'll implement a threshold-based approach. Courses with scores above this threshold will be considered relevant enough to recommend.

## Building the Recommendation Engine

Now, I'll implement a comprehensive recommendation system that calculates scores for all unknown courses for each test user in our dataset.

In [37]:
# Reload the datasets to ensure we're working with fresh data
test_users_df = pd.read_csv("data/ratings.csv")
profile_df = pd.read_csv("data/user_profile.csv")
course_genres_df = pd.read_csv("data/course_genre.csv")

# Create an empty dictionary to store recommendation results
res_dict = {}

Setting a recommendation threshold is crucial for filtering out low-confidence recommendations. After experimentation, I've determined an appropriate threshold value:

In [38]:
# Only recommend courses with scores above this threshold
# This value can be adjusted to control the number of recommendations
score_threshold = 95.0

Here's my implementation of the recommendation engine:

In [39]:
# Check the matrix dimensions for consistency
A = course_genres_df.iloc[:, 2:].values
b = profile_df[profile_df['user'] == 2].iloc[0, 1:].values
print(f"Course genre matrix shape: {A.shape}")
print(f"User profile vector shape: {b.shape}")
print(f"Resulting recommendation scores shape: {np.dot(A,b).shape}")

Course genre matrix shape: (307, 14)
User profile vector shape: (14,)
Resulting recommendation scores shape: (307,)


In [40]:
def generate_recommendation_scores():
    """
    Generate personalized recommendation scores for all test users across all unknown courses.
    
    The function iterates through each test user, identifies courses they haven't enrolled in,
    and calculates recommendation scores using the dot product between the user's profile
    vector and each course's genre vector.

    Returns:
    users (list): List of user IDs for which recommendations were generated.
    courses (list): List of recommended course IDs.
    scores (list): List of corresponding recommendation scores.
    """

    users = []      # Store user IDs
    courses = []    # Store recommended course IDs
    scores = []     # Store recommendation scores

    # Process each test user
    for user_id in test_user_ids:
        # Retrieve user profile
        test_user_profile = profile_df[profile_df['user'] == user_id]
        
        # Extract user interest vector
        test_user_vector = test_user_profile.iloc[0, 1:].values
        
        # Identify enrolled courses
        enrolled_courses = set(test_users_df[test_users_df['user'] == user_id]['item'].to_list())
        
        # Determine unknown courses (potential recommendations)
        unknown_courses = all_courses.difference(enrolled_courses)
        
        # Retrieve genre vectors for unknown courses
        unknown_course_df = course_genres_df[course_genres_df['COURSE_ID'].isin(unknown_courses)]
        unknown_course_ids = unknown_course_df['COURSE_ID'].values
        
        # Calculate recommendation scores using dot product
        recommendation_scores = np.dot(unknown_course_df.iloc[:, 2:].values, test_user_vector)
        
        # Filter and store recommendations
        for i in range(len(unknown_course_ids)):
            score = recommendation_scores[i]
            
            # Only keep high-confidence recommendations
            if score >= score_threshold:
                users.append(user_id)
                courses.append(unknown_course_ids[i])
                scores.append(score)

    return users, courses, scores

Now I'll execute the recommendation engine and analyze the results:

In [41]:
# Generate recommendations
users, courses, scores = generate_recommendation_scores()

# Organize results into a dictionary
res_dict = {
    'USER': users,
    'COURSE_ID': courses,
    'SCORE': scores
}

# Convert to DataFrame
res_df = pd.DataFrame(res_dict, columns=['USER', 'COURSE_ID', 'SCORE'])

# Save results to CSV
res_df.to_csv("profile_rs_results.csv", index=False)

# Display results
res_df

,USER,COURSE_ID,SCORE
0,2,GPXX0IBEN,105.0
1,2,GPXX0TY1EN,101.0
2,2,excourse02,95.0
3,2,excourse31,99.0
4,2,excourse72,136.0
...,...,...,...
1226,2046749,excourse73,117.0
1227,2079951,TMP0105EN,117.0
1228,2079951,excourse31,96.0
1229,2079951,excourse72,117.0


## Results Analysis

Let's analyze the effectiveness of our recommender system by examining two key metrics:

In [42]:
# Calculate average number of recommendations per user
avg_number_recs = res_df.groupby("USER").size().values.mean()
print(f"The average number of recommendations per user is {avg_number_recs:.2f}.")

The average number of recommendations per user is 5.55.


In [43]:
# Identify the most frequently recommended courses
top_courses = res_df.groupby("COURSE_ID").size().sort_values(ascending=False).iloc[0:10]

# Retrieve course titles
top_course_names = []
for course_id in top_courses.index:
    title = course_genres_df[course_genres_df["COURSE_ID"] == course_id]["TITLE"].to_string()
    top_course_names.append(title)
    
# Create a DataFrame for the top courses
top_courses_df = top_courses.to_frame(name="times_recommended").reset_index().rename(columns={"COURSE_ID": "course_id"})
top_courses_df.insert(1, "title", top_course_names)
top_courses_df

,course_id,title,times_recommended
0,excourse73,286 analyzing big data with sql,202
1,excourse72,285 foundations for big data analysis with sql,202
2,TMP0105EN,29 getting started with the data apache sp...,137
3,excourse31,244 cloud computing applications part 2 b...,86
4,GPXX0M6UEN,169 using the cql shell to execute keyspace...,53
5,excourse71,284 big data essentials hdfs mapreduce an...,53
6,excourse70,283 big data capstone project,53
7,excourse42,255 big data analysis hive spark sql dat...,53
8,excourse10,223 database architecture scale and nosql...,53
9,excourse05,218 \r\ndistributed computing with spark sql,53


The recommendation threshold plays a crucial role in balancing between recommendation quantity and quality. A lower threshold would yield more recommendations per user but with lower confidence, potentially overwhelming users with too many options. Conversely, a higher threshold ensures only the most relevant courses are recommended but may result in some users receiving few or no recommendations.

Based on my analysis, the current threshold value provides a good balance, keeping the average recommendations per user at a manageable level while ensuring high relevance.

## Conclusion

In this analysis, I've developed and implemented a content-based recommendation system for online courses. The approach uses linear algebra operations to model user interests as feature vectors and predict course relevance based on genre similarity.

The system demonstrates several strengths:

1. **Personalization**: By leveraging user profiles derived from past interactions, the system generates tailored recommendations aligned with individual interests.

2. **Scalability**: The approach efficiently handles a substantial user base and course catalog through optimized matrix operations.

3. **Interpretability**: The dot product method provides clear insight into why specific courses are recommended, as the scores directly correlate with shared interests between user profiles and course genres.

4. **Flexibility**: The adjustable threshold allows for fine-tuning the recommendation quantity and quality based on business requirements.

Future improvements could include:

- Incorporating collaborative filtering techniques for a hybrid approach
- Implementing more sophisticated similarity metrics beyond the dot product
- Adding dimensionality reduction techniques like PCA for handling larger feature spaces
- Developing a temporal component to account for evolving user interests over time

This implementation demonstrates how fundamental machine learning concepts can be applied to create practical recommendation systems that enhance user experience and engagement in online learning platforms.